# 05 — KNN e Métodos Baseados em Distância

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia técnico — KNN e Métodos Baseados em Distância](./reference_materials/05_knn.pdf)
- [Cheat sheet — KNN e Métodos Baseados em Distância](./cheat_sheets/cheat_sheet_knn.pdf)
- [Documentação e resultados](./README.pt-BR.md)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Explicar vizinhança, escolha de $k$, ponderação, escalonamento e diferença entre KNN classificador e regressor.

KNN é baseado em instâncias: na predição, procura exemplos próximos e agrega seus rótulos. `k` pequeno aumenta sensibilidade local/variância; `k` grande pode produzir fronteiras excessivamente suaves. Distâncias euclidiana/manhattan têm geometrias diferentes. Em dimensões muito altas as distâncias podem se tornar menos informativas.


## 2. Fundamentos matemáticos

$d_1(x,z)=\sum_j|x_j-z_j|$; $d_2(x,z)=\sqrt{\sum_j(x_j-z_j)^2}$. Na classificação, $\hat y(x)=\operatorname{moda}\{y_i:i\in N_k(x)\}$; para regressão, usa-se média (ou média ponderada) dos vizinhos.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
X,y=load_wine(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
cv=StratifiedKFold(5,shuffle=True,random_state=42)
for k in [1,3,7,15]:
    pipe=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=k))
    scores=cross_val_score(pipe,Xtr,ytr,cv=cv,scoring='balanced_accuracy')
    print(f'k={k:2d}: {scores.mean():.3f} ± {scores.std():.3f}')

k= 1: 0.982 ± 0.024
k= 3: 0.981 ± 0.015
k= 7: 0.968 ± 0.019
k=15: 0.975 ± 0.035


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.datasets import make_regression
from sklearn.metrics import mean_absolute_error
Xr,yr=make_regression(n_samples=160,n_features=3,noise=10,random_state=42)
a,b,c,d=train_test_split(Xr,yr,random_state=42,test_size=.25)
reg=make_pipeline(StandardScaler(),KNeighborsRegressor(n_neighbors=5,weights='distance')).fit(a,c)
print('MAE regressor KNN:',round(mean_absolute_error(d,reg.predict(b)),2))

MAE regressor KNN: 18.82


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: cálculo manual de 3 vizinhos mais próximos
import numpy as np
points=np.array([[0,0],[1,0],[0,2],[3,3]],float);labels=np.array([0,0,1,1]);query=np.array([.5,.4])
dists=np.linalg.norm(points-query,axis=1);neighbor_idx=np.argsort(dists)[:3]
print('Índices:',neighbor_idx.tolist(),'previsão:',np.bincount(labels[neighbor_idx]).argmax())

Índices: [0, 1, 2] previsão: 0


## 6. Interpretação, limitações e erros comuns

Nunca escolher `k` observando a métrica do teste. KNN exige cuidado com escalas e demanda memória/tempo de previsão quando há muitas observações. Empates, pontos raros e atributos irrelevantes influenciam resultados.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- `n_neighbors`: controla escala local.
- `weights="distance"` dá maior peso a vizinhos próximos.
- Escalonamento e validação devem estar no mesmo pipeline.
- Comparar desempenho com baseline e modelo paramétrico.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 03-knn.pdf; 01-introducao.pdf.

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** escala de distância. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **escala de distância** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **escala de distância** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


In [4]:
# Evidência tabular: compara k usando apenas o treino e validação cruzada.
from pathlib import Path
import pandas as pd
resultados_k = []
for valor_k in [1, 3, 7, 15]:
    est = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=valor_k))
    desempenho = cross_val_score(est, Xtr, ytr, cv=cv, scoring="balanced_accuracy")
    resultados_k.append({"k": valor_k, "balanced_accuracy_media": float(desempenho.mean()), "balanced_accuracy_desvio": float(desempenho.std())})
Path("outputs").mkdir(exist_ok=True)
pd.DataFrame(resultados_k).to_csv("outputs/05_comparacao_knn.csv", index=False)
print("Gerado: outputs/05_comparacao_knn.csv")

Gerado: outputs/05_comparacao_knn.csv
